#### Data Extraction File

##### Using Nasa's api to fetch Temprature, Humidity, Wind speed, Solar Radiation
##### Using pvlib to fetch hours of daylght
##### Combining both datasets into one
##### Solar Intensity dataset found at - check readme on github

In [11]:
import requests
import pandas as pd

In [12]:
#Phoenix, AZ
latitude = 33.4484
longitude = -112.0740

In [ ]:
#Api calls and fetching data
url = "https://power.larc.nasa.gov/api/temporal/daily/point"
params = {
    "parameters": "T2M,RH2M,WS2M,ALLSKY_SFC_SW_DWN",
    "start": "20060101",
    "end": "20061231",
    "latitude": latitude,
    "longitude": longitude,
    "format": "JSON",
    "community": "RE"
}

response = requests.get(url, params=params)
data = response.json()

raw_data = data['properties']['parameter']
df = pd.DataFrame({key: pd.Series(value) for key, value in raw_data.items()})
df.index = pd.to_datetime(df.index)

#Renaming columns for clarity
df.columns = ['Temperature_2m_C', 'Relative_Humidity_2m_%', 'Wind_Speed_2m_mps',
              'Solar_Radiation_MJ/m2']

print(df.head())

            Temperature_2m_C  Relative_Humidity_2m_%  Wind_Speed_2m_mps  \
2006-01-01             11.80                   44.51               1.20   
2006-01-02             14.83                   45.22               2.15   
2006-01-03             14.35                   48.32               1.03   
2006-01-04             13.51                   21.94               1.51   
2006-01-05             15.08                   20.54               3.29   

            Solar_Radiation_MJ/m2  
2006-01-01                   2.91  
2006-01-02                   2.79  
2006-01-03                   3.57  
2006-01-04                   3.55  
2006-01-05                   3.74  


In [14]:
import pvlib
from pvlib.location import Location
import pandas as pd

In [15]:
##Phoenix, AZ
lat, lon = 33.4484, -112.0740
site = Location(latitude=lat, longitude=lon)

In [ ]:
#Time range
times = pd.date_range('2006-01-01', '2006-12-31', freq='D', tz='US/Arizona')

#sunrise and sunset timings
sun_times = site.get_sun_rise_set_transit(times)

sun_times['daylight_hours'] = (sun_times['sunset'] - sun_times['sunrise']).dt.total_seconds() / 3600

print(sun_times[['sunrise', 'sunset', 'daylight_hours']].head())

# df.to_csv("C:/Users/tanma/Downloads/phoenix_2006_weather.csv")


                                                      sunrise  \
2006-01-01 00:00:00-07:00 2006-01-01 07:32:28.307304576-07:00   
2006-01-02 00:00:00-07:00 2006-01-02 07:32:39.575505920-07:00   
2006-01-03 00:00:00-07:00 2006-01-03 07:32:49.046702336-07:00   
2006-01-04 00:00:00-07:00 2006-01-04 07:32:56.703832576-07:00   
2006-01-05 00:00:00-07:00    2006-01-05 07:33:02.532960-07:00   

                                                       sunset  daylight_hours  
2006-01-01 00:00:00-07:00 2006-01-01 17:30:53.847637376-07:00        9.973761  
2006-01-02 00:00:00-07:00 2006-01-02 17:31:38.294657408-07:00        9.982978  
2006-01-03 00:00:00-07:00 2006-01-03 17:32:23.829279872-07:00        9.992995  
2006-01-04 00:00:00-07:00 2006-01-04 17:33:10.400733440-07:00       10.003805  
2006-01-05 00:00:00-07:00 2006-01-05 17:33:57.958767360-07:00       10.015396  


In [17]:
#fixing datasets to merge them

#Removing timezone info from sun_times
sun_times.index = sun_times.index.tz_localize(None)

df['daylight_hours'] = sun_times['daylight_hours']

In [18]:
df.head(5)

,Temperature_2m_C,Relative_Humidity_2m_%,Wind_Speed_2m_mps,Solar_Radiation_MJ/m2,daylight_hours
2006-01-01,11.80,44.51,1.20,2.91,9.973761
2006-01-02,14.83,45.22,2.15,2.79,9.982978
2006-01-03,14.35,48.32,1.03,3.57,9.992995
2006-01-04,13.51,21.94,1.51,3.55,10.003805
2006-01-05,15.08,20.54,3.29,3.74,10.015396


In [19]:
df.index.name = 'Date'
df.to_csv("C:/Users/tanma/Downloads/solar_panel_model/phoenix_2006_weather.csv")

In [20]:
df.head(5)

,Temperature_2m_C,Relative_Humidity_2m_%,Wind_Speed_2m_mps,Solar_Radiation_MJ/m2,daylight_hours
Date,,,,,
2006-01-01,11.80,44.51,1.20,2.91,9.973761
2006-01-02,14.83,45.22,2.15,2.79,9.982978
2006-01-03,14.35,48.32,1.03,3.57,9.992995
2006-01-04,13.51,21.94,1.51,3.55,10.003805
2006-01-05,15.08,20.54,3.29,3.74,10.015396
